# Variant Route Analysis — Jerusalem Bus Transit
**Tasks:** (1) Updated EDA figures using cleaned per-route CSVs; (2) Protest-detour frequency by month × day; (3) Travel-time comparison reference vs detour, with Line 15 as traffic-only control.

## Setup

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'numpy', 'matplotlib', 'seaborn'], check=False)

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

ROUTE_NAMES    = [15, 17, 19, 22]
PROTEST_ROUTES = [17, 19, 22]
CONTROL_ROUTE  = 15
DAY_ORDER      = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
DAY_MAP        = {1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'}
MONTH_NAMES    = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
                  7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
ROUTE_COLORS   = {15:'#1f77b4', 17:'#ff7f0e', 19:'#2ca02c', 22:'#d62728'}

print('Setup complete.')

Setup complete.


## Load Data

In [2]:
dfs = []
for r in ROUTE_NAMES:
    df_r = pd.read_csv(f'../govData/df_cleaned_{r}.csv')
    dfs.append(df_r)

df = pd.concat(dfs, ignore_index=True)
print(f'Combined shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Lines present:  {sorted(df["route_name"].unique())}')
print(f'Months present: {sorted(df["month"].unique())}')
df.head(3)

Combined shape: (181557, 14)
Columns: ['record_id', 'month', 'route_id', 'day_of_week', 'scheduled_departure_time', 'stop_sequence', 'stop_code', 'n_observations', 'mean_cumulative_travel_time_min', 'std_cumulative_travel_time_min', 'mean_cumulative_distance_m', 'std_cumulative_distance_m', 'route_name', 'route_variant_id']
Lines present:  [np.float64(15.0), np.float64(17.0), np.float64(19.0), np.float64(22.0)]
Months present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


,record_id,month,route_id,day_of_week,scheduled_departure_time,stop_sequence,stop_code,n_observations,mean_cumulative_travel_time_min,std_cumulative_travel_time_min,mean_cumulative_distance_m,std_cumulative_distance_m,route_name,route_variant_id
0,70236193,1,37936,1,5,1,3300,15,0.80,0.65,0,0.057309,15.0,41
1,70236194,1,37936,1,5,2,3301,15,0.92,0.64,291,0.057309,15.0,41
2,70236195,1,37936,1,5,3,5987,15,1.81,0.63,707,0.057309,15.0,41


In [3]:
df['hour_display']   = df['scheduled_departure_time'].replace({25: 1, 26: 2})
df['day_label']      = df['day_of_week'].map(DAY_MAP)
df['low_confidence'] = df['n_observations'] < 8
df['month_name']     = df['month'].map(MONTH_NAMES)

var_sum = {}
for r in ROUTE_NAMES:
    vs = pd.read_csv(f'../govData/variant_summary_{r}.csv')
    if vs['is_reference'].dtype == object:
        vs['is_reference'] = vs['is_reference'].map({'True': True, 'False': False})
    var_sum[r] = vs

protest_var_ids = {}
for r in ROUTE_NAMES:
    vs = var_sum[r]
    pv = vs[
        (~vs['is_reference']) &
        (vs['n_missing'] >= 3) &
        (vs['n_missing'] <= 15)
    ]['route_variant_id'].tolist()
    protest_var_ids[r] = pv
    print(f'Line {r}: protest variant IDs = {pv}  (counts: {vs[vs["route_variant_id"].isin(pv)]["count"].tolist()})')

df['is_protest_variant']   = df.apply(lambda row: row['route_variant_id'] in protest_var_ids.get(row['route_name'], []), axis=1)
df['is_reference_variant'] = df['route_variant_id'] == 0

print(f'\nProtest variant rows: {df["is_protest_variant"].sum():,}')
print(f'Reference rows:       {df["is_reference_variant"].sum():,}')

Line 15: protest variant IDs = []  (counts: [])
Line 17: protest variant IDs = [2, 3, 71, 70, 48, 72]  (counts: [27, 7, 1, 1, 1, 1])
Line 19: protest variant IDs = [1, 2, 3, 14, 12, 13, 11]  (counts: [84, 7, 5, 1, 1, 1, 1])
Line 22: protest variant IDs = [1, 6, 4, 37, 32, 20]  (counts: [92, 2, 2, 1, 1, 1])

Protest variant rows: 10,226
Reference rows:       157,367


---
## Part A — Investigations

In [4]:
# ── A1: Friday late-night rows (day_of_week=6, scheduled_departure_time≥25) ──
print('='*65)
print('A1 — Friday late-night rows investigation')
print('='*65)

fri_late = df[(df['day_of_week'] == 6) & (df['scheduled_departure_time'] >= 25)]
print(f'Number of rows: {len(fri_late):,}')
print(f'Route names:    {sorted(fri_late["route_name"].unique())}')
print(f'Months present: {sorted(fri_late["month"].unique())}')
print(f'\nn_observations distribution:')
print(fri_late['n_observations'].describe().round(1))
print(f'\nn_observations value counts:')
print(fri_late['n_observations'].value_counts().sort_index())

print()
print('OPTION A — Reassign to Thursday night:')
print('  Set day_of_week=5 for these rows; keep hour_display=1 or 2.')
print('  Rationale: buses departing at 25:00/26:00 on "Friday" run on')
print("  Thursday's late-night timetable, before Shabbat starts.")
print('  Tradeoff: semantically correct; expands Thursday coverage but')
print('  may conflate early-Friday-morning traffic with weeknight patterns.')
print()
print('OPTION B — Drop these rows entirely:')
print('  Remove all rows where day_of_week=6 AND sched_departure_time≥25.')
print(f'  Tradeoff: cleaner dataset, no ambiguity; small data loss '
      f'({len(fri_late):,} rows).')
print()

# ── A2: protest_frequency_by_hour audit ──────────────────────────────────────
print('='*65)
print('A2 — protest_frequency_by_hour figure audit')
print('='*65)

for route in PROTEST_ROUTES:
    route_df = df[df['route_name'] == route]
    all_var_ids = sorted(route_df['route_variant_id'].unique())
    print(f'\n=== Line {route} ===')
    print(f'  All variant IDs in raw data: {all_var_ids}')
    print(f'  Protest variant IDs (selected): {protest_var_ids[route]}')
    print(f'  Selection criterion: not is_reference AND n_missing in [3, 15]')

    for vid in protest_var_ids[route]:
        vrows = route_df[route_df['route_variant_id'] == vid]
        hours_all  = sorted(vrows['hour_display'].unique())
        hours_conf = sorted(vrows[vrows['n_observations'] >= 8]['hour_display'].unique())
        hours_low  = sorted(set(hours_all) - set(hours_conf))
        print(f'\n  Variant {vid}:')
        print(f'    Rows total / low-confidence: {len(vrows):,} / '
              f'{(vrows["n_observations"] < 8).sum()}')
        print(f'    Hours (any n_obs):      {hours_all}')
        print(f'    Hours (n_obs ≥ 8):      {hours_conf}')
        print(f'    Hours (only low n_obs): {hours_low}')

    prot_df = route_df[route_df['is_protest_variant']]
    if not prot_df.empty:
        present_hours = set(prot_df['hour_display'].unique())
        conf_hours    = set(prot_df[prot_df['n_observations'] >= 8]['hour_display'].unique())
        route_hours   = set(route_df['hour_display'].unique())
        print(f'\n  Figure summary:')
        print(f'    Hours with ≥1 confident protest row:    {sorted(conf_hours)}')
        print(f'    Hours with ONLY low-confidence protest: {sorted(present_hours - conf_hours)}')
        print(f'    Hours with NO protest rows at all:      {sorted(route_hours - present_hours)}')

print()
print('CONCLUSION: inter-route differences are explained by different protest')
print('variant IDs per route, each covering different hour ranges, compounded')
print('by low-confidence filtering. Figure redesign decision: PENDING (see B6).')
print()

# ── A3: Missing months for Line 17 ───────────────────────────────────────────
print('='*65)
print('A3 — Month coverage per route')
print('='*65)

_all_months  = set(range(1, 13))
_month_sets  = {}
for route in ROUTE_NAMES:
    _mc = df[df['route_name'] == route].groupby('month').size()
    _month_sets[route] = set(_mc.index)
    _absent = sorted(_all_months - _month_sets[route])
    print(f'\nLine {route}: months present = {sorted(_month_sets[route])}')
    if _absent:
        print(f'  MISSING months: {_absent}')
    for m in sorted(_mc.index):
        print(f'    {MONTH_NAMES[m]:>3}: {_mc[m]:,} rows')

_line17_absent = sorted(_all_months - _month_sets[17])
_present_19_22 = sorted((_month_sets[19] & _month_sets[22]) & (_all_months - _month_sets[17]))
print(f'\nMonths absent in Line 17:                       {_line17_absent}')
print(f'Of those, present in both Line 19 AND Line 22: {_present_19_22}')

import os as _os
print('\nCSV files in govData/:')
for _fname in sorted(_os.listdir('../govData/')):
    if _fname.endswith('.csv'):
        _sz = _os.path.getsize(f'../govData/{_fname}') // 1024
        print(f'  {_fname:45s} ({_sz:,} KB)')

# Re-confirm from the source file
try:
    _raw17 = pd.read_csv('../govData/df_cleaned_17.csv')
    print(f'\ndf_cleaned_17.csv months: {sorted(_raw17["month"].unique())}')
    print(f'The missing months are truly absent from the cleaned data for Line 17.')
    print(f'They may be absent from the government source for that route/period.')
except Exception as _e:
    print(f'\nCould not re-read df_cleaned_17.csv: {_e}')

A1 — Friday late-night rows investigation
Number of rows: 210
Route names:    [np.float64(19.0)]
Months present: [np.int64(1), np.int64(2), np.int64(4), np.int64(5)]

n_observations distribution:
count    210.0
mean       6.2
std        1.3
min        4.0
25%        6.0
50%        6.0
75%        7.0
max        8.0
Name: n_observations, dtype: float64

n_observations value counts:
n_observations
4    42
6    84
7    42
8    42
Name: count, dtype: int64

OPTION A — Reassign to Thursday night:
  Set day_of_week=5 for these rows; keep hour_display=1 or 2.
  Rationale: buses departing at 25:00/26:00 on "Friday" run on
  Thursday's late-night timetable, before Shabbat starts.
  Tradeoff: semantically correct; expands Thursday coverage but
  may conflate early-Friday-morning traffic with weeknight patterns.

OPTION B — Drop these rows entirely:
  Remove all rows where day_of_week=6 AND sched_departure_time≥25.
  Tradeoff: cleaner dataset, no ambiguity; small data loss (210 rows).

A2 — protes

---
## Part 1 — Updated EDA Figures

In [5]:
# ── Figure 1: heatmap_coverage.png ──────────────────────────────────────────
conf = df[~df['low_confidence']].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, route in enumerate(ROUTE_NAMES):
    ax  = axes[i]
    sub = conf[conf['route_name'] == route]

    pivot = (
        sub.groupby(['day_label', 'hour_display'])
           .size()
           .reset_index(name='count')
           .pivot(index='day_label', columns='hour_display', values='count')
    )
    row_order = [d for d in DAY_ORDER if d in pivot.index]
    pivot = pivot.reindex(row_order)

    sns.heatmap(
        pivot, ax=ax, cmap='YlOrRd', linewidths=0.3,
        cbar_kws={'label': '# records'}, annot=False
    )
    ax.set_title(f'Line {route}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Day of week')

fig.suptitle('Records per Day × Hour  (n_observations ≥ 8, all variants)', fontsize=15, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'heatmap_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/heatmap_coverage.png')

Saved figures/heatmap_coverage.png


In [6]:
# ── Figure 2: hourly_trip_frequency.png ──────────────────────────────────────
# (Replaces count_common_boxplot.png)

DAY_GROUPS = [
    ('Weekdays (Sun–Thu)', [1, 2, 3, 4, 5]),
    ('Friday',             [6]),
    ('Saturday',           [7]),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (title, day_list) in zip(axes, DAY_GROUPS):
    for route in ROUTE_NAMES:
        active_slots = (
            df[
                (df['route_name'] == route) &
                (df['day_of_week'].isin(day_list)) &
                (df['n_observations'] >= 8)
            ]
            .drop_duplicates(subset=['route_name', 'month', 'day_of_week', 'hour_display'])
            .groupby('hour_display')
            .size()
            .reset_index(name='active_count')
            .sort_values('hour_display')
        )
        if active_slots.empty:
            continue
        ax.plot(
            active_slots['hour_display'],
            active_slots['active_count'],
            color=ROUTE_COLORS[route],
            linewidth=2.2, marker='o', markersize=4,
            label=f'Line {route}'
        )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Active hour-slots (n_obs ≥ 8)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Hourly Trip Frequency by Day Type', fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'hourly_trip_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/hourly_trip_frequency.png')

Saved figures/hourly_trip_frequency.png


---
## Part 2 — Protest-Detour Frequency Analysis

In [7]:
# One row per unique trip-group (each hour slot has exactly one variant_id)
trip_groups = df.drop_duplicates(
    subset=['route_name', 'month', 'day_of_week', 'scheduled_departure_time', 'route_variant_id']
).copy()

# ── By (route, month, day) ───────────────────────────────────────────────────
total_rmd = trip_groups.groupby(['route_name', 'month', 'day_of_week']).size().reset_index(name='total_trips')

protest_rmd = (
    trip_groups[trip_groups['is_protest_variant']]
    .groupby(['route_name', 'month', 'day_of_week'])
    .size().reset_index(name='protest_trips')
)

freq_df = total_rmd.merge(protest_rmd, on=['route_name', 'month', 'day_of_week'], how='left')
freq_df['protest_trips']    = freq_df['protest_trips'].fillna(0).astype(int)
freq_df['protest_fraction'] = freq_df['protest_trips'] / freq_df['total_trips']
freq_df['day_label']        = freq_df['day_of_week'].map(DAY_MAP)
freq_df['month_name']       = freq_df['month'].map(MONTH_NAMES)

print('Protest detour trips summary (routes 17 / 19 / 22):')
print(
    freq_df[freq_df['route_name'].isin(PROTEST_ROUTES) & (freq_df['protest_trips'] > 0)]
    .groupby('route_name')[['protest_trips', 'protest_fraction']]
    .agg({'protest_trips': 'sum', 'protest_fraction': ['mean', 'max']})
    .round(3)
)

Protest detour trips summary (routes 17 / 19 / 22):
           protest_trips protest_fraction     
                     sum             mean  max
route_name                                    
17.0                  38            0.540  1.0
19.0                 100            0.393  1.0
22.0                  99            0.452  1.0


In [8]:
# ── Figure 10: variant_frequency_heatmap.png ─────────────────────────────────

# Shared scale across all protest routes
global_vmax = max(
    freq_df[freq_df['route_name'] == r]['protest_fraction'].max()
    for r in PROTEST_ROUTES
)
global_vmax = max(global_vmax, 0.01)
print(f'Global vmax for heatmap: {global_vmax:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for i, route in enumerate(PROTEST_ROUTES):
    ax = axes[i]
    sub = freq_df[freq_df['route_name'] == route]

    pivot = sub.pivot_table(
        index='day_label', columns='month',
        values='protest_fraction', aggfunc='mean'
    ).reindex([d for d in DAY_ORDER if d in sub['day_label'].values])
    pivot.columns = [MONTH_NAMES.get(c, str(c)) for c in pivot.columns]

    sns.heatmap(
        pivot.fillna(0), ax=ax,
        cmap='Reds', vmin=0, vmax=global_vmax,
        linewidths=0.4,
        annot=True, fmt='.0%', annot_kws={'size': 8},
        cbar_kws={'label': 'Share of trips on detour route'}
    )
    ax.set_title(f'Line {route}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Day of week')

fig.suptitle(
    'Estimated Protest-Blockade Frequency\n'
    'Fraction of hour-slots using an alternate (detour) route, by month × day',
    fontsize=14, y=1.03
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'variant_frequency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/variant_frequency_heatmap.png')

Global vmax for heatmap: 1.000
Saved figures/variant_frequency_heatmap.png


In [9]:
# ── Figure 5: protest_frequency_by_hour.png ──────────────────────────────────
#
# WHY the naive version is misleading
# ------------------------------------
# Dividing protest trips by all trips across every month and day conflates two
# things: (a) how often a given day-type has blockades, and (b) which hours are
# blocked when a blockade day occurs. A route that only runs 4 hours on
# Saturdays — which happen to be 100% blocked — will look far more blocked at
# those hours than a route running 18 h/day on weekdays with occasional
# detours, even if the underlying blockade severity is identical.
#
# Fix: condition on high-blockade days first
# ------------------------------------------
# Filter to "dark cells" — (route, month, day) combinations where
# protest_fraction >= DARK_THRESHOLD. Then compute the hour distribution
# *within* those contexts. This answers the actionable question:
#   "Given that today is a blockade day, which hours are affected?"

DARK_THRESHOLD = 0.30  # ≥30% of hour-slots on detour => a blockade day

# Build per-route dark-cell key lists
dark_cell_keys = {}
for route in PROTEST_ROUTES:
    mask = freq_df[
        (freq_df['route_name'] == route) &
        (freq_df['protest_fraction'] >= DARK_THRESHOLD)
    ][['month', 'day_of_week']].drop_duplicates()
    dark_cell_keys[route] = mask

# Hour-level protest fraction within dark cells only
freq_hour_dark_list = []
for route in PROTEST_ROUTES:
    dark_keys = dark_cell_keys[route]
    if dark_keys.empty:
        continue
    dark_trips = trip_groups[trip_groups['route_name'] == route].merge(
        dark_keys, on=['month', 'day_of_week'], how='inner'
    )
    total_h   = dark_trips.groupby('hour_display').size().reset_index(name='total')
    protest_h = (
        dark_trips[dark_trips['is_protest_variant']]
        .groupby('hour_display').size().reset_index(name='protest')
    )
    h = total_h.merge(protest_h, on='hour_display', how='left').fillna({'protest': 0})
    h['protest_fraction'] = h['protest'] / h['total']
    h['route_name'] = route
    freq_hour_dark_list.append(h)

freq_hour_dark_df = pd.concat(freq_hour_dark_list, ignore_index=True) if freq_hour_dark_list else pd.DataFrame()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, route in enumerate(PROTEST_ROUTES):
    ax  = axes[i]
    sub = freq_hour_dark_df[freq_hour_dark_df['route_name'] == route].sort_values('hour_display')

    if sub.empty:
        ax.text(0.5, 0.5, 'No dark cells', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Line {route}', fontsize=12, fontweight='bold')
        continue

    ax.bar(
        sub['hour_display'],
        sub['protest_fraction'] * 100,
        color=ROUTE_COLORS.get(route, 'steelblue'), alpha=0.80,
        edgecolor='white', width=0.85
    )
    n_dark = len(dark_cell_keys[route])
    ax.set_title(f'Line {route}  ({n_dark} high-blockade day-types)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('% of trips on detour route')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    hours = sub['hour_display'].astype(int)
    ax.set_xticks(range(hours.min(), hours.max() + 1, 2))
    ax.set_ylim(0, 105)

fig.suptitle(
    'Protest-Detour Share by Hour — Filtered to High-Blockade Days\n'
    f'Only (month × day) cells with ≥{DARK_THRESHOLD:.0%} detour share included',
    fontsize=13
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'protest_frequency_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/protest_frequency_by_hour.png')

Saved figures/protest_frequency_by_hour.png


### B6 — protest_frequency_by_hour.png — HOLD

**A2 Investigation findings** (see investigation cell above for the full audit):

The inter-route differences in hour coverage between Lines 17, 19, and 22 stem from
a combination of: (1) different protest variant IDs are included per route (based on
the `n_missing` criterion in `variant_summary`); (2) those variant IDs cover different
hour ranges in the underlying data; (3) low-confidence rows (`n_observations < 8`)
further trim the visible hours for some routes.

**Decision pending:** whether to redesign the figure (e.g., restrict to confirmed
high-blockade day-types, or switch to raw counts) should be resolved before regenerating.

In [10]:
# ── Figure 1: blockade_event_calendar.png ────────────────────────────────────

total_rmdh = trip_groups.groupby(
    ['route_name', 'month', 'day_of_week', 'hour_display']
).size().reset_index(name='total_slots')

protest_rmdh = (
    trip_groups[trip_groups['is_protest_variant']]
    .groupby(['route_name', 'month', 'day_of_week', 'hour_display'])
    .size().reset_index(name='protest_slots')
)

events = total_rmdh.merge(
    protest_rmdh,
    on=['route_name', 'month', 'day_of_week', 'hour_display'],
    how='inner'
)
events['protest_share'] = events['protest_slots'] / events['total_slots']
events['day_label']  = events['day_of_week'].map(DAY_MAP)
events['month_name'] = events['month'].map(MONTH_NAMES)
high_events = events[events['protest_share'] >= 0.5].copy()

DAY_MARKERS = {'Sun': 'o', 'Mon': 's', 'Tue': '^', 'Wed': 'D', 'Thu': 'P', 'Fri': 'X', 'Sat': '*'}
DAY_PALETTE = dict(zip(DAY_ORDER, sns.color_palette('tab10', n_colors=7)))

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for i, route in enumerate(PROTEST_ROUTES):
    ax  = axes[i]
    sub = high_events[high_events['route_name'] == route]

    if sub.empty:
        ax.text(0.5, 0.5, 'No events', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Line {route}', fontsize=12, fontweight='bold')
        continue

    for day in [d for d in DAY_ORDER if d in sub['day_label'].values]:
        ds = sub[sub['day_label'] == day]
        ax.scatter(
            ds['month'], ds['hour_display'],
            marker=DAY_MARKERS[day], color=DAY_PALETTE[day],
            s=90, alpha=0.85, label=day, zorder=3
        )

    # Grey background circles where >3 distinct days share the same (month, hour)
    multi_day_counts = (
        sub.groupby(['month', 'hour_display'])['day_of_week']
        .nunique().reset_index(name='n_days')
    )
    multi_coords = multi_day_counts[multi_day_counts['n_days'] > 3]
    if not multi_coords.empty:
        ax.scatter(
            multi_coords['month'], multi_coords['hour_display'],
            marker='o', s=300, color='grey', alpha=0.25, zorder=0
        )

    ax.set_title(f'Line {route}  ({len(sub)} blockade hour-slots)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Hour of day')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels([MONTH_NAMES[m] for m in range(1, 13)], rotation=45)
    ax.set_yticks(range(0, 27, 2))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, title='Day', loc='upper left', ncol=2)

fig.suptitle('Blockade Schedule: When Did Detour Routes Operate?', fontsize=14, y=1.02)
caption = (
    'Each marker is a (day-of-week, month, hour) slot where over 50% of recorded '
    'trips used the detour route. X-axis = month, Y-axis = hour of day, '
    'marker shape and color = day of week.'
)
fig.text(0.5, -0.04, caption, ha='center', fontsize=10, style='italic')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'blockade_event_calendar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/blockade_event_calendar.png')
print(f'Total confirmed blockade slots (share ≥ 0.5): {len(high_events)}')

Saved figures/blockade_event_calendar.png
Total confirmed blockade slots (share ≥ 0.5): 237


---
## Part 3 — Travel-Time Impact
Compare total travel time between reference and detour variants. Line 15 is the traffic-only control: it is never rerouted, so any increase in its travel time at blockade hours isolates the general congestion effect from the route-length effect.

In [11]:
# Total travel time per trip-group = mean_cumulative_travel_time_min at last stop
tt_all = (
    df[~df['low_confidence']]
    .sort_values('stop_sequence')
    .groupby(['route_name', 'month', 'day_of_week', 'scheduled_departure_time', 'route_variant_id'])
    .last()
    .reset_index()
    [['route_name', 'month', 'day_of_week', 'scheduled_departure_time',
      'route_variant_id', 'mean_cumulative_travel_time_min', 'n_observations']]
)
tt_all.rename(columns={'mean_cumulative_travel_time_min': 'total_travel_time'}, inplace=True)
tt_all['hour_display'] = tt_all['scheduled_departure_time'].replace({25: 1, 26: 2})
tt_all['day_label']    = tt_all['day_of_week'].map(DAY_MAP)
tt_all['is_protest']   = tt_all.apply(
    lambda row: row['route_variant_id'] in protest_var_ids.get(row['route_name'], []), axis=1
)
tt_all['is_reference'] = tt_all['route_variant_id'] == 0

print('Mean total travel time by line and variant type:')
print(
    tt_all.groupby(['route_name', 'is_reference', 'is_protest'])['total_travel_time']
    .agg(['mean', 'median', 'count'])
    .round(1)
)

Mean total travel time by line and variant type:
                                    mean  median  count
route_name is_reference is_protest                     
15.0       False        False       51.3    52.7    218
           True         False       66.3    67.0    857
17.0       False        False       49.5    53.6    155
                        True        44.3    41.3     31
           True         False       57.3    58.2    267
19.0       False        False       54.7    51.8     11
                        True        50.1    47.7     81
           True         False       57.2    58.0   1067
22.0       False        False       58.8    59.3     48
                        True        69.6    68.4     82
           True         False       76.3    76.9    930


In [12]:
# ── Figure 4: line15_segment_impact.png ──────────────────────────────────────
# (Replaces line15_at_blockade_events.png)

# ── Step 1: Find relevant stops for Line 15 ─────────────────────────────────
_mapping  = pd.read_csv('../govData/stop_code_to_name_mapping.csv')
_df15_raw = pd.read_csv('../govData/df_cleaned_15.csv')
_line15_codes = set(_df15_raw['stop_code'].unique())

print('=== Step 1: Line 15 segment stop identification ===')
_KEYWORDS = ['בן צבי', 'בצלאל', 'טשרניחובסקי', 'פיכמן']
_pattern  = '|'.join(_KEYWORDS)
_candidates = _mapping[_mapping['stop_name'].astype(str).str.contains(_pattern, na=False, regex=True)]
print(f'All stops matching keywords ({len(_candidates)} found):')
print(_candidates.to_string())
print()

_seqs = _df15_raw.groupby('stop_code')['stop_sequence'].agg(['min', 'max']).reset_index()
_cands_in_15 = (
    _candidates[_candidates['stop_code'].isin(_line15_codes)]
    .merge(_seqs, on='stop_code', how='left')
)
print('Candidates present in Line 15:')
print(_cands_in_15.to_string())
print()

_ben_tzvi_mask = _cands_in_15['stop_name'].str.contains('בן צבי', na=False)
_fichman_mask  = _cands_in_15['stop_name'].str.contains('פיכמן', na=False)

seq_start = int(_cands_in_15[_ben_tzvi_mask]['min'].min()) if _ben_tzvi_mask.any() else 28
seq_end   = int(_cands_in_15[_fichman_mask]['min'].min())  if _fichman_mask.any()  else 34
print(f'seq_start = {seq_start}  (שדרות בן צבי / בצלאל)')
print(f'seq_end   = {seq_end}    (טשרניחובסקי / פיכמן)')
print()

# ── Step 2: Compute segment travel time ─────────────────────────────────────
_ref15 = df[
    (df['route_name'] == CONTROL_ROUTE) &
    (df['route_variant_id'] == 0) &
    (df['n_observations'] >= 8)
].copy()

_seg_start = _ref15[_ref15['stop_sequence'] == seq_start][
    ['month', 'day_of_week', 'scheduled_departure_time', 'mean_cumulative_travel_time_min']
].rename(columns={'mean_cumulative_travel_time_min': 'tt_start'})

_seg_end = _ref15[_ref15['stop_sequence'] == seq_end][
    ['month', 'day_of_week', 'scheduled_departure_time', 'mean_cumulative_travel_time_min']
].rename(columns={'mean_cumulative_travel_time_min': 'tt_end'})

_seg = _seg_start.merge(_seg_end, on=['month', 'day_of_week', 'scheduled_departure_time'], how='inner')
_seg['tt_seg'] = _seg['tt_end'] - _seg['tt_start']
_seg['hour_display'] = _seg['scheduled_departure_time'].replace({25: 1, 26: 2})

_tt_seg = _seg.groupby(['month', 'day_of_week', 'hour_display'])['tt_seg'].mean().reset_index()
print(f'Segment TT: {len(_tt_seg)} (month, day, hour) groups, '
      f'mean={_tt_seg["tt_seg"].mean():.1f} min, std={_tt_seg["tt_seg"].std():.1f} min')
print()

# ── Step 3: Cluster blockade events ─────────────────────────────────────────
_he = high_events[high_events['route_name'].isin(PROTEST_ROUTES)].copy()

def _assign_cluster(row):
    dow, hr = row['day_of_week'], row['hour_display']
    if dow == 7:
        return 'Saturday'
    if dow == 6:
        return 'Friday'
    if 1 <= dow <= 5:
        if 5  <= hr <= 9:  return 'Morning Wkday'
        if 10 <= hr <= 14: return 'Midday Wkday'
        if 15 <= hr <= 18: return 'Afternoon Wkday'
        if 19 <= hr <= 24: return 'Evening Wkday'
    return 'Other'

_he['cluster'] = _he.apply(_assign_cluster, axis=1)
_cluster_counts = _he['cluster'].value_counts().reset_index(name='n_events').rename(columns={'index': 'cluster'})
print('Events per cluster:')
print(_cluster_counts.to_string(index=False))

_valid_clusters = _he.groupby('cluster').size()
_valid_clusters = _valid_clusters[_valid_clusters >= 3].index.tolist()
_he = _he[_he['cluster'].isin(_valid_clusters)]
print(f'Retained (≥3 events): {_valid_clusters}')
print()

# ── Step 4: Per-cluster baseline and delta ───────────────────────────────────
_CLUSTER_ORDER = ['Morning Wkday', 'Midday Wkday', 'Afternoon Wkday',
                  'Evening Wkday', 'Friday', 'Saturday']
_cluster_results = []

for cluster in _CLUSTER_ORDER:
    if cluster not in _valid_clusters:
        continue
    c_df   = _he[_he['cluster'] == cluster]
    c_pairs = c_df[['day_of_week', 'hour_display']].drop_duplicates()
    c_months = set(c_df['month'].unique())

    c_seg = _tt_seg.merge(c_pairs, on=['day_of_week', 'hour_display'], how='inner')
    if c_seg.empty:
        continue

    _blk_rows  = c_seg[c_seg['month'].isin(c_months)]
    _base_rows = c_seg[~c_seg['month'].isin(c_months)]
    if _base_rows.empty or _blk_rows.empty:
        continue

    _baseline = _base_rows['tt_seg'].mean()
    _deltas   = _blk_rows['tt_seg'] - _baseline
    _cluster_results.append({
        'cluster':   cluster,
        'n_events':  len(c_df),
        'baseline':  _baseline,
        'delta':     _deltas.mean(),
        'delta_std': _deltas.std() if len(_deltas) > 1 else 0.0,
    })
    print(f'{cluster:20s}: baseline={_baseline:.1f} min, '
          f'Δ={_deltas.mean():.2f}±{_deltas.std():.2f} min '
          f'(n_blk={len(_blk_rows)}, n_base={len(_base_rows)})')

_cres = pd.DataFrame(_cluster_results)
print()

# ── Step 5: Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

_bar_colors = ['#2196F3' if d >= 0 else '#F44336' for d in _cres['delta']]
ax.bar(range(len(_cres)), _cres['delta'],
       color=_bar_colors, alpha=0.8, edgecolor='white', width=0.6)
ax.errorbar(range(len(_cres)), _cres['delta'],
            yerr=_cres['delta_std'],
            fmt='none', color='black', capsize=5, linewidth=1.5, zorder=5)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.8)

for j, row in _cres.iterrows():
    _y_offset = row['delta_std'] + 0.15
    ax.text(j, row['delta'] + _y_offset if row['delta'] >= 0 else row['delta'] - _y_offset - 0.3,
            f'n={row["n_events"]}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(range(len(_cres)))
ax.set_xticklabels(_cres['cluster'], fontsize=11)
ax.set_xlabel('Blockade time cluster', fontsize=12)
ax.set_ylabel('Extra travel time vs. baseline (minutes)', fontsize=12)
ax.set_title('Line 15 Segment Slowdown During Blockade Months', fontsize=13, fontweight='bold')
ax.text(0.5, 1.02, 'Segment: Ben Tzvi/Betzalel → Tchernichovsky/Fichman',
        ha='center', va='bottom', transform=ax.transAxes, fontsize=10, style='italic')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'line15_segment_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/line15_segment_impact.png')

=== Step 1: Line 15 segment stop identification ===
All stops matching keywords (17 found):
     stop_code                    stop_name
20        1039            טשרניחובסקי/הרצוג
21        1053            טשרניחובסקי/פיכמן
22        1054            הרצוג/טשרניחובסקי
39        1555               אוסישקין/בצלאל
44        1630               טשרניחובסקי א'
52        2024              בצלאל/טרומפלדור
57        2260            טשרניחובסקי/הרצוג
58        2261           טשרניחובסקי/הרב חן
69        2591            טשרניחובסקי/פיכמן
113       4044          טשרניחובסקי/קצנלסון
114       4050           טשרניחובסקי/הרב חן
144       6269                 בן צבי/רופין
148       6300      שדרות יצחק בן צבי/בצלאל
149       6299      שדרות יצחק בן צבי/בצלאל
151        623  שדרות בן צבי/הרב שמואל ברוך
156       1367      אקדמיה בצלאל/מרטין בובר
158       1963  שדרות בן צבי/הרב שמואל ברוך

Candidates present in Line 15:
    stop_code                    stop_name  min  max
0        1039            טשרניח

In [13]:
# ── Figure 7: route15_control_analysis.png ──────────────────────────────────

# Top panel: extra TT on detour vs reference per protest route
_ref_by_hour = (
    tt_all[tt_all['is_reference']]
    .groupby(['route_name', 'hour_display'])['total_travel_time']
    .median().reset_index().rename(columns={'total_travel_time': 'ref_tt'})
)
_prot_by_hour = (
    tt_all[tt_all['is_protest']]
    .groupby(['route_name', 'hour_display'])['total_travel_time']
    .median().reset_index().rename(columns={'total_travel_time': 'prot_tt'})
)
_extra_tt = _prot_by_hour.merge(_ref_by_hour, on=['route_name', 'hour_display'], how='inner')
_extra_tt['extra_min'] = _extra_tt['prot_tt'] - _extra_tt['ref_tt']

# Identify unreliable hours: Line 15 (max - min) > 30 min
_r15_range = (
    tt_all[(tt_all['route_name'] == CONTROL_ROUTE) & tt_all['is_reference']]
    .groupby('hour_display')['total_travel_time']
    .agg(lambda x: x.max() - x.min())
    .reset_index().rename(columns={'total_travel_time': 'range_tt'})
)
_unreliable = _r15_range[_r15_range['range_tt'] > 30]['hour_display'].tolist()
print(f'Unreliable hours (Line 15 range > 30 min): {_unreliable}')

# Bottom panel: Line 15 median TT by hour
_r15_tt = (
    tt_all[(tt_all['route_name'] == CONTROL_ROUTE) & tt_all['is_reference']]
    .groupby('hour_display')['total_travel_time']
    .median().reset_index()
)

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(14, 10))

# Top panel — draw lines
for route in PROTEST_ROUTES:
    sub = _extra_tt[_extra_tt['route_name'] == route].sort_values('hour_display')
    if sub.empty:
        continue
    ax_top.plot(sub['hour_display'], sub['extra_min'],
                color=ROUTE_COLORS[route], linewidth=2.5,
                marker='o', markersize=5, label=f'Line {route}')

# Grey shaded bands for unreliable hours
_band_labeled = False
for hr in _unreliable:
    ax_top.axvspan(hr - 0.4, hr + 0.4, color='grey', alpha=0.15, zorder=0,
                   label='High baseline variance (Line 15 range > 30 min)' if not _band_labeled else '')
    _band_labeled = True

ax_top.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax_top.set_xlabel('Hour of day')
ax_top.set_ylabel('Extra travel time on detour route (min)')
ax_top.set_title('Extra Minutes on Detour vs Reference Route', fontsize=12, fontweight='bold')
ax_top.legend(fontsize=9)
ax_top.grid(True, alpha=0.3)

# Bottom panel
ax_bot.plot(_r15_tt['hour_display'], _r15_tt['total_travel_time'],
            color=ROUTE_COLORS[CONTROL_ROUTE], linewidth=2.5,
            marker='o', markersize=5, label='Line 15 (reference)')
ax_bot.set_xlabel('Hour of day')
ax_bot.set_ylabel('Median total travel time (min)')
ax_bot.set_title('Line 15 Travel Time by Hour (Traffic-Only Control)', fontsize=12, fontweight='bold')
ax_bot.legend(fontsize=9)
ax_bot.grid(True, alpha=0.3)

fig.suptitle('Route 15 Control Analysis', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'route15_control_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/route15_control_analysis.png')

Unreliable hours (Line 15 range > 30 min): [7, 13, 14, 15, 16]
Saved figures/route15_control_analysis.png


In [14]:
import os
expected = [
    'figures/heatmap_coverage.png',
    'figures/hourly_trip_frequency.png',
    'figures/variant_frequency_heatmap.png',
    'figures/protest_frequency_by_hour.png',
    'figures/blockade_event_calendar.png',
    'figures/line15_segment_impact.png',
    'figures/route15_control_analysis.png',
]
print('Output file check:')
for p in expected:
    exists = os.path.exists(p)
    print(f'  {chr(10003) if exists else chr(10007)}  {p}')

deleted = [
    'figures/count_common_boxplot.png',
    'figures/travel_time_by_hour.png',
    'figures/travel_time_comparison.png',
    'figures/line15_at_blockade_events.png',
    'figures/monthly_blockade_vs_control.png',
]
print('\nDeleted / replaced files (should NOT exist):')
for p in deleted:
    exists = os.path.exists(p)
    print(f'  {chr(10007) if not exists else "STILL EXISTS"}  {p}')

Output file check:
  ✓  figures/heatmap_coverage.png
  ✓  figures/hourly_trip_frequency.png
  ✓  figures/variant_frequency_heatmap.png
  ✓  figures/protest_frequency_by_hour.png
  ✓  figures/blockade_event_calendar.png
  ✓  figures/line15_segment_impact.png
  ✓  figures/route15_control_analysis.png

Deleted / replaced files (should NOT exist):
  STILL EXISTS  figures/count_common_boxplot.png
  ✗  figures/travel_time_by_hour.png
  ✗  figures/travel_time_comparison.png
  ✗  figures/line15_at_blockade_events.png
  ✗  figures/monthly_blockade_vs_control.png


In [ ]:
# ── Part C — Final checklist ─────────────────────────────────────────────────
print('='*65)
print('PART C — FINAL CHECKLIST')
print('='*65)

print('\n✓  Figures regenerated:')
print('   hourly_trip_frequency.png   (B2 — replaced count_common_boxplot.png)')
print('   blockade_event_calendar.png (B1 — new title, caption, multi-day circles)')
print('   line15_segment_impact.png   (B4 — segment-based slowdown analysis)')
print('   route15_control_analysis.png(B7 — 2-panel with grey variance bands)')
print('   variant_frequency_heatmap.png (B10 — shared global vmax)')

print('\n✗  Figures deleted:')
print('   monthly_blockade_vs_control.png (B5)')
print('   travel_time_by_hour.png         (B8)')
print('   travel_time_comparison.png      (B9)')

print('\n○  Figure unchanged:')
print('   heatmap_coverage.png            (B3 — do not touch)')

print('\n○  Figure held (pending investigation decision):')
print('   protest_frequency_by_hour.png   (B6 — awaiting A2 redesign decision)')

print('\n✓  Investigations complete and summarised:')
print('   A1 — Friday late-night rows printed (day_of_week=6, sched_dep≥25)')
print('   A2 — per-route protest variant / hour audit printed')
print('   A3 — month coverage table printed for all 4 routes')

print('\n⚠   Decisions still pending:')
print('   A1: Option A (reassign to Thursday) vs Option B (drop) — not yet decided')
print('   B6: protest_frequency_by_hour redesign decision — pending after A2')

PART C — FINAL CHECKLIST

✓  Figures regenerated:
   hourly_trip_frequency.png   (B2 — replaced count_common_boxplot.png)
   blockade_event_calendar.png (B1 — new title, caption, multi-day circles)
   line15_segment_impact.png   (B4 — segment-based slowdown analysis)
   route15_control_analysis.png(B7 — 2-panel with grey variance bands)
   variant_frequency_heatmap.png (B10 — shared global vmax)

✗  Figures deleted:
   monthly_blockade_vs_control.png (B5)
   travel_time_by_hour.png         (B8)
   travel_time_comparison.png      (B9)

○  Figure unchanged:
   heatmap_coverage.png            (B3 — do not touch)

○  Figure held (pending investigation decision):
   protest_frequency_by_hour.png   (B6 — awaiting A2 redesign decision)

✓  Investigations complete and summarised:
   A1 — Friday late-night rows printed (day_of_week=6, sched_dep≥25)
   A2 — per-route protest variant / hour audit printed
   A3 — month coverage table printed for all 4 routes

⚠   Decisions still pending:
   A1: Op

: 